In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


class RetrievalEngine:

    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents):
        """
        documents: list[str]
        """
        self.documents.extend(documents)

        new_embeddings = self.model.encode(documents)

        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([
                self.embeddings,
                new_embeddings
            ])

    def search(self, question, k=3):
        if len(self.documents) == 0:
            raise Exception("No documents found.")

        query_embedding = self.model.encode(question)

        scores = cosine_similarity(
            [query_embedding],
            self.embeddings
        )[0]

        top_indices = np.argsort(scores)[::-1][:k]

        results = []

        for idx in top_indices:
            results.append({
                "document": self.documents[idx],
                "score": float(scores[idx])
            })

        return results

    def count(self):
        return len(self.documents)

    def clear(self):
        self.documents = []
        self.embeddings = None

In [3]:
engine = RetrievalEngine()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
engine.add_documents([
    "Spiderman is Peter Parker.",
    "Batman lives in Gotham.",
    "Ironman owns Stark Industries.",
    "Thor is the God of Thunder."
])

print(engine.count())

4


In [ ]:
result = engine.search(
    "Who made MARK II?",
    k=3
)

for r in result:
    print(r)

{'document': 'Spiderman is Peter Parker.', 'score': 0.5161482691764832}
{'document': 'Thor is the God of Thunder.', 'score': 0.5058257579803467}
{'document': 'Ironman owns Stark Industries.', 'score': 0.5000253915786743}
